# TRACE方法对比

  TRACE 方法对比
- 源脚本：`experiments/03_证伪实验/scripts/运行_TRACE方法对比.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准

这本 notebook 对应 TRACE 方法和论文主线方案之间的直接对比。

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/03_证伪实验/scripts/运行_TRACE方法对比.py
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""在同一批复杂问答上比较 TRACE 方法与本文主线方案。"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.common import DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, ensure_dir
from retrieval_pipeline.datasets import load_crud_cases
from retrieval_pipeline.metrics import evaluate_crud_results
from retrieval_pipeline.pipeline import PipelineVariant, RagExperimentPipeline

OUTPUT_ROOT = ROOT / "results" / "05_TRACE方法对比"
CRUD_SUBSET_SPLITS = {"questanswer_2docs": 797, "questanswer_3docs": 797}

VARIANTS = (
    PipelineVariant(
        key="ours_router_4_6",
        label="Ours（分项重排 + 覆盖取证 + 按题作答）",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="task_router",
        multi_snippet_count=1,
        final_source_count=4,
        complex_source_count=6,
        selection_mode="aspect_cover_v2",
    ),
    PipelineVariant(
        key="trace_method_4_6",
        label="TRACE 方法（透明三段式生成）",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="trace_structured",
        multi_snippet_count=1,
        final_source_count=4,
        complex_source_count=6,
        selection_mode="aspect_cover_v2",
    ),
)


def run_variant_parallel(
    pipeline: RagExperimentPipeline,
    prepared,
    variant: PipelineVariant,
    *,
    workers: int,
) -> list[Any]:
    total = len(prepared.cases)
    if workers <= 1:
        results = []
        for index, case in enumerate(prepared.cases, start=1):
            if index == 1 or index % 10 == 0 or index == total:
                print(f"[crud-trace-method] {variant.key}: {index}/{total}", flush=True)
            results.append(pipeline.run_case(prepared, case, variant))
        return results

    ordered_results: list[Any] = [None] * total
    completed = 0
    with ThreadPoolExecutor(max_workers=workers) as executor:
        future_to_index = {
            executor.submit(pipeline.run_case, prepared, case, variant): index
            for index, case in enumerate(prepared.cases)
        }
        for future in as_completed(future_to_index):
            index = future_to_index[future]
            ordered_results[index] = future.result()
            completed += 1
            if completed == 1 or completed % 10 == 0 or completed == total:
                print(f"[crud-trace-method] {variant.key}: {completed}/{total}", flush=True)
    return ordered_results


def write_outputs(
    output_root: Path,
    metric_rows: list[dict[str, Any]],
    summaries: list[dict[str, Any]],
    detail_rows: list[dict[str, Any]],
    manifest: dict[str, Any],
) -> None:
    (output_root / "TRACE对比_汇总.json").write_text(
        json.dumps({"summaries": summaries}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (output_root / "TRACE对比_指标表.json").write_text(
        json.dumps(metric_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (output_root / "TRACE对比_逐题明细.json").write_text(
        json.dumps(detail_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    with (output_root / "TRACE对比_指标表.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "variant",
                "label",
                "sample_count",
                "retrieval_hit_rate_at_1",
                "retrieval_hit_rate_at_3",
                "integration_string_similarity",
                "integration_focus_f1",
                "integration_quality",
                "complex_quality",
                "latency_p50_ms",
                "latency_p95_ms",
            ],
        )
        writer.writeheader()
        writer.writerows(metric_rows)
    (output_root / "评测批次说明.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description="在同一批 CRUD 子样本评测批次上对比 TRACE 方法与 Ours。")
    parser.add_argument("--embedding-model", default=DEFAULT_EMBEDDING_MODEL)
    parser.add_argument("--llm-model", default=DEFAULT_LLM_MODEL)
    parser.add_argument("--qa-2doc-samples", type=int, default=20)
    parser.add_argument("--qa-3doc-samples", type=int, default=20)
    parser.add_argument("--distractor-count", type=int, default=600)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=3)
    parser.add_argument(
        "--output-root",
        default="",
        help="可选输出目录；为空时写入 results/05_TRACE方法对比/TRACE方法对比_20260520/",
    )
    args = parser.parse_args()

    output_root = (
        Path(args.output_root).resolve() if args.output_root else OUTPUT_ROOT / "TRACE方法对比_20260520"
    )
    ensure_dir(output_root)
    cache_root = ensure_dir(ROOT / ".cache")

    cases, docs = load_crud_cases(
        summary_samples=0,
        qa_1doc_samples=0,
        qa_2doc_samples=args.qa_2doc_samples,
        qa_3doc_samples=args.qa_3doc_samples,
        hallu_samples=0,
        negative_samples=0,
        distractor_count=args.distractor_count,
        seed=args.seed,
    )
    expected_count = args.qa_2doc_samples + args.qa_3doc_samples
    if len(cases) != expected_count:
        raise RuntimeError(f"CRUD 子样本当前评测批次数异常，期望 {expected_count}，实际 {len(cases)}")

    pipeline = RagExperimentPipeline(
        cache_root=cache_root,
        embedding_model=args.embedding_model,
        llm_model=args.llm_model,
    )
    prepared = pipeline.prepare_dataset(
        "crud_trace_method_compare_batch",
        cases,
        docs,
        include_contextual=False,
        include_parent_child=False,
        include_query_rewrite=False,
    )

    summaries: list[dict[str, Any]] = []
    detail_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    manifest = {
        "dataset": "crud_trace_method_compare_batch",
        "crud_subset_total": sum(CRUD_SUBSET_SPLITS.values()),
        "crud_subset_splits": CRUD_SUBSET_SPLITS,
        "evaluation_batch_size": len(cases),
        "evaluation_batch_splits": {
            "questanswer_2docs": args.qa_2doc_samples,
            "questanswer_3docs": args.qa_3doc_samples,
        },
        "embedding_model": args.embedding_model,
        "llm_model": args.llm_model,
        "distractor_count": args.distractor_count,
        "seed": args.seed,
        "workers": args.workers,
        "case_ids": [case.case_id for case in cases],
        "variants": [variant.label for variant in VARIANTS],
    }

    for variant in VARIANTS:
        results = run_variant_parallel(pipeline, prepared, variant, workers=max(1, args.workers))
        evaluation = evaluate_crud_results(
            variant.key,
            results,
            cases,
            ragas_case_ids=(),
            qa_ragas_case_ids=(),
            multidoc_ragas_case_ids=(),
            enable_ragas=False,
            semantic_model_name=args.embedding_model,
        )
        summary = dict(evaluation.summary)
        summary["label"] = variant.label
        summaries.append(summary)
        detail_rows.extend(evaluation.detail_rows)
        metric_rows.append(
            {
                "variant": variant.key,
                "label": variant.label,
                "sample_count": summary["sample_count"],
                "retrieval_hit_rate_at_1": summary["retrieval_hit_rate_at_1"],
                "retrieval_hit_rate_at_3": summary["retrieval_hit_rate_at_3"],
                "integration_string_similarity": summary["integration_string_similarity"],
                "integration_focus_f1": summary["integration_focus_f1"],
                "integration_quality": summary["integration_quality"],
                "complex_quality": summary["complex_quality"],
                "latency_p50_ms": summary["latency_p50_ms"],
                "latency_p95_ms": summary["latency_p95_ms"],
            }
        )
        write_outputs(output_root, metric_rows, summaries, detail_rows, manifest)

    print(json.dumps({"output_root": str(output_root), "metric_rows": metric_rows}, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 评测批次说明

- 文件：`../results/05_TRACE方法对比/TRACE方法对比_20260520/评测批次说明.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/05_TRACE方法对比/TRACE方法对比_20260520/评测批次说明.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "dataset": "crud_trace_method_compare_batch",
  "crud_subset_total": 1594,
  "crud_subset_splits": {
    "questanswer_2docs": 797,
    "questanswer_3docs": 797
  },
  "evaluation_batch_size": 40,
  "evaluation_batch_splits": {
    "questanswer_2docs": 20,
    "questanswer_3docs": 20
  },
  "embedding_model": "bge-m3:latest",
  "llm_model": "qwen2.5:7b-instruct",
  "distractor_count": 600,
  "seed": 42,
  "workers": 3,
  "case_ids": [
    "questanswer_2docs_001",
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_004",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_007",
    "questanswer_2docs_008",
    "questanswer_2docs_009",
    "questanswer_2docs_010",
    "questanswer_2docs_011",
    "questanswer_2docs_012",
    "questanswer_2docs_013",
    "questanswer_2docs_014",
    "questanswer_2docs_015",
    "questanswer_2docs_016",
    "questanswer_2docs_017",
    "questanswer_2docs_018",
    "questanswer_2docs_019",
    "qu

### TRACE 对比汇总

- 文件：`../results/05_TRACE方法对比/TRACE方法对比_20260520/TRACE对比_汇总.json`

In [2]:
from pathlib import Path
import json

path = Path('../results/05_TRACE方法对比/TRACE方法对比_20260520/TRACE对比_汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "summaries": [
    {
      "variant": "ours_router_4_6",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.9,
      "retrieval_hit_rate_at_3": 0.975,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "qa_similarity": 0.8668,
      "qa_string_similarity": 0.5138,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
      "overall_similarity": 0.8668,
      "overall_string_similarity": 0.5138,
      "summary_similarity": 0.0,
      "summary_string_similarity": 0.0,
      "noise_robustness": 0.0,
      "negati

### TRACE 对比指标表（JSON）

- 文件：`../results/05_TRACE方法对比/TRACE方法对比_20260520/TRACE对比_指标表.json`

In [3]:
from pathlib import Path
import json

path = Path('../results/05_TRACE方法对比/TRACE方法对比_20260520/TRACE对比_指标表.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


[
  {
    "variant": "ours_router_4_6",
    "label": "Ours（分项重排 + 覆盖取证 + 按题作答）",
    "sample_count": 40,
    "retrieval_hit_rate_at_1": 0.9,
    "retrieval_hit_rate_at_3": 0.975,
    "integration_string_similarity": 0.5138,
    "integration_focus_f1": 0.5809,
    "integration_quality": 0.6751,
    "complex_quality": 0.6751,
    "latency_p50_ms": 3557.47,
    "latency_p95_ms": 7832.38
  },
  {
    "variant": "trace_method_4_6",
    "label": "TRACE 方法（透明三段式生成）",
    "sample_count": 40,
    "retrieval_hit_rate_at_1": 0.9,
    "retrieval_hit_rate_at_3": 0.975,
    "integration_string_similarity": 0.4656,
    "integration_focus_f1": 0.4886,
    "integration_quality": 0.647,
    "complex_quality": 0.647,
    "latency_p50_ms": 7438.45,
    "latency_p95_ms": 8980.31
  }
]


### TRACE 对比指标表（CSV）

- 文件：`../results/05_TRACE方法对比/TRACE方法对比_20260520/TRACE对比_指标表.csv`

In [4]:
from pathlib import Path
import csv

path = Path('../results/05_TRACE方法对比/TRACE方法对比_20260520/TRACE对比_指标表.csv')
with path.open('r', encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print(f'rows={len(rows)} preview={min(len(rows), 8)}')
for row in rows[:8]:
    print(row)


variant,label,sample_count,retrieval_hit_rate_at_1,retrieval_hit_rate_at_3,integration_string_similarity,integration_focus_f1,integration_quality,complex_quality,latency_p50_ms,latency_p95_ms
ours_router_4_6,Ours（分项重排 + 覆盖取证 + 按题作答）,40,0.9,0.975,0.5138,0.5809,0.6751,0.6751,3557.47,7832.38
trace_method_4_6,TRACE 方法（透明三段式生成）,40,0.9,0.975,0.4656,0.4886,0.647,0.647,7438.45,8980.31
